## Tekmir Data Science Intern Challenge

#### Author: Nicole Huynh; nthuynhbc@gmail.com

### Track 1: Fictional Domain Packet

### Part 1: Setup and Data Cleaning/Exploration

In [38]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [39]:
data = pd.read_csv('product_usage_events.csv')
data.head(5)

,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
0,2026-08-01,Sales,Lead summary,email,42,35,29,3,8.5,0.74,4.1,normal day
1,2026-08-01,Sales,Lead summary,manual,18,12,8,2,6.0,0.61,3.8,normal day
2,2026-08-01,Support,Reply draft,queue,55,48,39,6,4.5,0.82,4.0,normal day
3,2026-08-01,Support,Reply draft,manual,11,8,5,1,3.0,0.68,NaN,missing rating
4,2026-08-01,Product,Feedback clustering,csv upload,12,9,6,2,14.0,0.59,3.6,small sample


In [40]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                41 non-null     object 
 1   team                41 non-null     object 
 2   workflow            41 non-null     object 
 3   source              41 non-null     object 
 4   sessions            41 non-null     int64  
 5   completed           41 non-null     int64  
 6   accepted_output     41 non-null     int64  
 7   flagged_for_review  41 non-null     int64  
 8   avg_minutes_saved   41 non-null     float64
 9   median_confidence   40 non-null     float64
 10  user_rating         40 non-null     float64
 11  notes               41 non-null     object 
dtypes: float64(3), int64(4), object(5)
memory usage: 4.0+ KB


In [41]:
data['date'] = pd.to_datetime(data['date']) # convert str to datetime object

data = data.drop(25) # drop duplicate data row, as per notes

sales = data[data['team'] == 'Sales'].drop(columns='team')
support = data[data['team'] == 'Support'].drop(columns='team')
product = data[(data['team'] == 'Product') | (data['team'] == 'product')].drop(columns='team') # account for different casing in team name

In [42]:
sales.head()

,date,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
0,2026-08-01,Lead summary,email,42,35,29,3,8.5,0.74,4.1,normal day
1,2026-08-01,Lead summary,manual,18,12,8,2,6.0,0.61,3.8,normal day
6,2026-08-02,Lead summary,email,46,38,31,3,8.2,0.76,4.2,normal day
7,2026-08-02,Lead summary,manual,20,14,9,2,6.4,0.63,3.9,normal day
12,2026-08-03,Lead summary,email,50,40,32,4,8.4,0.77,4.2,normal day


**Notes:** A new prompt version began on `2026-08-04`, so we should split the data into two separate dataframes. I'll do this for this and each of the other teams' data.

In [43]:
support

,date,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
2,2026-08-01,Reply draft,queue,55,48,39,6,4.5,0.82,4.0,normal day
3,2026-08-01,Reply draft,manual,11,8,5,1,3.0,0.68,NaN,missing rating
8,2026-08-02,Reply draft,queue,61,52,41,7,4.2,0.83,4.1,normal day
9,2026-08-02,Reply draft,manual,12,9,6,1,3.2,0.69,3.7,normal day
14,2026-08-03,Reply draft,queue,64,54,42,8,4.0,0.84,4.0,normal day
15,2026-08-03,Reply draft,manual,13,10,6,1,3.4,0.70,3.8,normal day
20,2026-08-04,Reply draft,queue,69,57,45,9,4.1,0.86,4.2,new prompt version started
21,2026-08-04,Reply draft,manual,14,10,7,1,3.5,0.72,3.9,new prompt version started
27,2026-08-05,Reply draft,queue,72,60,47,9,4.2,0.87,4.2,normal day
28,2026-08-05,Reply draft,manual,16,12,8,2,3.4,0.73,3.8,normal day


**Notes:** The review policy changed mid-day on `2026-08-07`, so we will not look at the `flagged_for_review` column for that row anymore as it will not be consistent with the review standards for the rest of this dataframe.

In [44]:
product.head()

,date,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
4,2026-08-01,Feedback clustering,csv upload,12,9,6,2,14.0,0.59,3.6,small sample
5,2026-08-01,Feedback clustering,manual,5,4,3,1,11.0,0.55,3.5,small sample
10,2026-08-02,Feedback clustering,csv upload,16,11,8,2,13.5,0.60,3.7,normal day
11,2026-08-02,Feedback clustering,manual,6,4,2,1,10.5,0.52,3.4,team casing differs
16,2026-08-03,Feedback clustering,csv upload,18,12,8,2,13.0,0.61,3.8,normal day


**Notes:** Noted the missing confidence in row with idx 30. "Team casing differs" is notes in row with idx 11, but I am unsure what this means. 

I will delete the columns `workflow` and `notes` to clean up the data as well as using `pd.get_dummies` to convert the categorical `'source'` column into integers I can work with, and then split each team's data into the old and new prompt versions for better analysis.

In [45]:
def clean(df):
    df = pd.concat([df, pd.get_dummies(df['source'], dtype=int)], axis=1)
    return df.drop(columns=['workflow', 'source', 'notes'])

In [46]:
data = clean(data)

sales = clean(sales)
support = clean(support)
support = support.drop(38) # drop last row as review standards changed

product = clean(product)

sales.head()

,date,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,email,manual
0,2026-08-01,42,35,29,3,8.5,0.74,4.1,1,0
1,2026-08-01,18,12,8,2,6.0,0.61,3.8,0,1
6,2026-08-02,46,38,31,3,8.2,0.76,4.2,1,0
7,2026-08-02,20,14,9,2,6.4,0.63,3.9,0,1
12,2026-08-03,50,40,32,4,8.4,0.77,4.2,1,0


In [47]:
sales_old = sales[:6]
sales_new = sales[6:]

supp_old = support[:6]
supp_new = support[6:]

prod_old = product[:6]
prod_new = product[6:]

In [48]:
sales_old.head()

,date,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,email,manual
0,2026-08-01,42,35,29,3,8.5,0.74,4.1,1,0
1,2026-08-01,18,12,8,2,6.0,0.61,3.8,0,1
6,2026-08-02,46,38,31,3,8.2,0.76,4.2,1,0
7,2026-08-02,20,14,9,2,6.4,0.63,3.9,0,1
12,2026-08-03,50,40,32,4,8.4,0.77,4.2,1,0


### Part 2: Analysis

For my primary question, I would like to report on which team's workflow is the "best". But, "best" is a loose and subjective term -- what would it mean for a workplace like SignalDesk? Primarily, I believe that a workflow is "good" if it is *helpful* -- it should truly reduce the valuable time an employee or team spends on menial or mundane tasks while producing quality work. After all, if a workflow or tool requires lots of human review, it can actually create more work for a human reviewer rather than reducing their load. 

First, I want to examine the number of **completed** outputs out of all the sessions for each team (new and old prompts). I also want to calculate, of the completed outputs, how many were **accepted** and how many were **flagged**. I'll be paying attention especially to the `flagged_for_review` column and any stats I generate from this, since the main goal is to find the workflow that best reduces human input/work. Finally, I would like to check how confident the model is, given its performance. After all, what use is an AI tool or assistant that is overconfident in their work?

In [49]:
def calc_stats(df):
    # calculate per-day, per-input stats as columns and prints 
    # summary of stats for entire dataframe

    # completed outputs over number of sessions run, as a float
    df['percent_output'] = df['completed'] / df['sessions']

    # percentage of completed outputs accepted and flagged, respectively
    df['percent_accepted'] = df['accepted_output'] / df['completed']
    df['percent_flagged'] = df['flagged_for_review'] / df['completed']

    total_sessions = sum(df['sessions'])
    total_completed = sum(df['completed'])
    total_accepted = sum(df['accepted_output'])
    total_flagged = sum(df['flagged_for_review'])

    return (df.round(2), round(total_completed/total_sessions, 2), round(total_accepted/total_completed, 2), round(total_flagged/total_completed, 2))

In [50]:
team_stats = {'Sales': calc_stats(sales), 'Product': calc_stats(product), 'Support': calc_stats(support)}

In [51]:
def print_stats(stats):
    for k,v in stats.items():
        print(f"{k} -- averages completed outputs per session: {v[1]}.")
        print(f"{k} -- percentage of user-accepted outputs: {v[2]}.")
        print(f"{k} -- percentage of user-flagged outputs is {v[3]}.", end='\n\n')

In [52]:
print_stats(team_stats)

Sales -- averages completed outputs per session: 0.81.
Sales -- percentage of user-accepted outputs: 0.82.
Sales -- percentage of user-flagged outputs is 0.08.

Product -- averages completed outputs per session: 0.67.
Product -- percentage of user-accepted outputs: 0.66.
Product -- percentage of user-flagged outputs is 0.19.

Support -- averages completed outputs per session: 0.82.
Support -- percentage of user-accepted outputs: 0.77.
Support -- percentage of user-flagged outputs is 0.14.



Because the workflows changed their prompts on `2026-08-04`, let's take a look at the summary stats for each team pre- and post- new prompt, as well as the performance of the new prompt overall for all 3 teams.

In [53]:
old_data = data[data['date'] < pd.to_datetime('2026-08-04')]
new_data = data[data['date'] >= pd.to_datetime('2026-08-04')]

all_data = {'Old Prompt': calc_stats(old_data), 'New Prompt': calc_stats(new_data)}

print_stats(all_data)

Old Prompt -- averages completed outputs per session: 0.79.
Old Prompt -- percentage of user-accepted outputs: 0.76.
Old Prompt -- percentage of user-flagged outputs is 0.13.

New Prompt -- averages completed outputs per session: 0.78.
New Prompt -- percentage of user-accepted outputs: 0.78.
New Prompt -- percentage of user-flagged outputs is 0.13.



Overall, the new prompt does not seem to be much more helpful across all three teams. However, how does the prompt help each team's workflow?

In [54]:
stats_by_prompt = {
    'Sales, old': calc_stats(sales_old),
    'Sales, new': calc_stats(sales_new),
    'Product, old': calc_stats(prod_old),
    'Product, new': calc_stats(prod_new),
    'Support, old': calc_stats(supp_old),
    'Support, new': calc_stats(supp_new),
    }

In [55]:
print_stats(stats_by_prompt)

Sales, old -- averages completed outputs per session: 0.78.
Sales, old -- percentage of user-accepted outputs: 0.77.
Sales, old -- percentage of user-flagged outputs is 0.1.

Sales, new -- averages completed outputs per session: 0.82.
Sales, new -- percentage of user-accepted outputs: 0.85.
Sales, new -- percentage of user-flagged outputs is 0.06.

Product, old -- averages completed outputs per session: 0.7.
Product, old -- percentage of user-accepted outputs: 0.67.
Product, old -- percentage of user-flagged outputs is 0.2.

Product, new -- averages completed outputs per session: 0.65.
Product, new -- percentage of user-accepted outputs: 0.66.
Product, new -- percentage of user-flagged outputs is 0.18.

Support, old -- averages completed outputs per session: 0.84.
Support, old -- percentage of user-accepted outputs: 0.77.
Support, old -- percentage of user-flagged outputs is 0.13.

Support, new -- averages completed outputs per session: 0.81.
Support, new -- percentage of user-accepted

For each team but **Sales**, the new prompt lead to an increase in the percentage of flagged outputs. With this data, I would recommend **Product** and **Support** to keep their old prompts in their workflows, while **Sales** should use the new prompt. Overall, when we examine the user-accepted and user-flagged outputs, as well as average # of completed outputs per session, it seems that **Sales'** workflow using the new prompt is the best.

Across both prompts, **Sales** also had the workflow that yielded the best average stats using the above metrics.

### Part 3: Model Overconfidence

To verify that the **Sales** team's workflow is indeed the best, I also wanted to make sure that the model's median confidence was generally lower than the actual percentage of accepted outputs. Intuitively, if a model's confidence is consistently higher than the actual percentage of its user-accepted outputs, this model is overly certain in its outputs and is not that helpful.

In [56]:
def examine_confidence(df):
    # returns the number of instances where model's median confidence was higher than 
    # the percentage of accepted completed outputs by 0.02
    return sum((team_stats[df][0]['median_confidence'] - team_stats[df][0]['percent_accepted']) > 0.02)

In [57]:
[examine_confidence(df) for df in ['Sales', 'Product', 'Support']]

[0, 1, 10]

Of the 13 rows of **Support**'s complete workflow data, 11 rows report a median model confidence that is much higher than the percentage of user-accepted outputs! I would advise the **Support** team to reconsider using their workflow.